# Corpus-regression sweep analysis

In [350]:
import polars as pl

from src import get_repo_base
from src.data.corpus_regression import candidate_lookforward_tokens
from src.experiments.corpus_regression.analysis import (
    CorpusRegressionAnalysisConfig,
    plot_methods_pred_std_vs_eval,
    plot_methods_vs_eval,
    plot_methods_vs_lookforward,
)
from src.experiments.corpus_regression.config import artifacts_dir

ARTIFACTS = artifacts_dir()

# Print summarize() tables in full (default polars truncation hides columns).
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(64)
pl.Config.set_fmt_str_lengths(80)

polars.config.Config

## Parameters

In [351]:
# === Parameters ===
NUM_SAMPLES: int = 500_000       # 100_000 or 500_000
NUM_LOOKFORWARD_TOKENS: int = 1  # K=1 (NTP), K=2,..,8 (skip-ahead). Per-method
                                 # plots show this single look; pass a list
                                 # to group_by it (e.g. =[1,2,4,8]).
GAUSSIAN_STDEV: float = 1.0      # 1.0 or 0.5
LABEL_TYPE: str = "token_id"   # "rademacher" or "token_id"
NORMALIZE_LABELS: bool = True   # True for [0,1]-normalized token_id labels
LABEL_RANGE: tuple[float, float] = (0.0, 1.0)  # target range when normalized
BATCH_SIZE: int = 256             # Training batch size. 64 is the default and
                                 # produces no `bs-{N}` path segment (byte-for-
                                 # byte backward-compatible). Pass a list to
                                 # group_by it (e.g. =[64, 128]).
LR_PER_SAMPLE: float = 2e-5     # per-sample LR; resolves to head_lr = LR_PER_SAMPLE × batch_size
LR_SCHEDULE: str = "cosine"        # "flat" (today's behaviour) or "cosine" (linear warmup → half-cosine decay)
WARMUP_RATIO: float = 0.05       # fraction of train_steps for warmup; ignored when LR_SCHEDULE == "flat"
LR_MIN_RATIO: float = 0.5        # cosine end LR as fraction of peak; ignored when LR_SCHEDULE == "flat"
TRAIN_FROM_SCRATCH: bool = True  # True -> read from artifacts_dir()/<method>_scratch/
TRAIN_STEPS: int = 1_500        # Must match the --train-steps used by the sweep
                                 # scripts (10_000 for the checked-in sweeps;
                                 # vince-sweep.bash uses 1_500). Pass a list
                                 # to group_by it.

# Derived output path — namespaced by combo to avoid overwriting
_combo_slug = f"n{NUM_SAMPLES // 1000}k_sigma{GAUSSIAN_STDEV}"
if LABEL_TYPE != "rademacher":
    _combo_slug += f"_{LABEL_TYPE}"
if NORMALIZE_LABELS:
    _combo_slug += "_normalized"
_combo_slug += f"_lr{LR_PER_SAMPLE:.0e}"
# Encode batch size only when non-default (mirrors the `bs-{N}` artifact-path
# segment convention: bs=64 is omitted, anything else appears).
if BATCH_SIZE != 64:
    _combo_slug += f"_bs{BATCH_SIZE}"
# Encode the LR schedule so a cosine sweep doesn't clobber the flat sweep's
# WRITEUP_ASSETS html files (or vice versa) when both are run side-by-side.
if LR_SCHEDULE != "flat":
    _combo_slug += f"_sched-{LR_SCHEDULE}_warm-{WARMUP_RATIO:.3f}_min-{LR_MIN_RATIO:.2f}"
if TRAIN_FROM_SCRATCH:
    _combo_slug += "_scratch"
# Tag the train_steps budget so a 1.5k-steps sweep doesn't clobber the
# 10k-steps HTML in WRITEUP_ASSETS (or vice versa).
_combo_slug += f"_steps-{TRAIN_STEPS:06d}"
# Tag the look value so multiple looks don't clobber each other's HTML.
_combo_slug += f"_look{NUM_LOOKFORWARD_TOKENS}"
WRITEUP_ASSETS = get_repo_base() / "writeup" / "assets" / "corpus-regression" / _combo_slug

## Supervised-learning

In [352]:
sl = CorpusRegressionAnalysisConfig.from_sl_mse_sweep(
    artifacts_root=ARTIFACTS,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,
    lr_per_sample=LR_PER_SAMPLE,
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
)
if sl is None:
    print("SL: no artifacts")
else:
    sl.describe("SL")
    print(sl.summarize(metric="corr"))
    print(sl.summarize(metric="mse"))
    display(
        sl.plot_vs_eval(
            "corr",
            title="SL: per-eval (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_eval_corr.html",
        )
    )
    display(
        sl.plot_vs_eval(
            "mse",
            title="SL: per-eval (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_eval_mse.html",
        )
    )

SL                                   1 runs     1 groups   up to 1 seeds/group
shape: (1, 8)
┌────────┬─────────────┬───────────┬─────────┬─────────────┬─────────────┬────────────┬────────────┐
│ study  ┆ num_lookfor ┆ best_step ┆ n_seeds ┆ train_corr_ ┆ val_corr_me ┆ train_pred ┆ val_pred_s │
│ ---    ┆ ward        ┆ ---       ┆ ---     ┆ mean        ┆ an          ┆ _std_ratio ┆ td_ratio_m │
│ str    ┆ ---         ┆ i64       ┆ i64     ┆ ---         ┆ ---         ┆ _mean      ┆ ean        │
│        ┆ i64         ┆           ┆         ┆ f64         ┆ f64         ┆ ---        ┆ ---        │
│        ┆             ┆           ┆         ┆             ┆             ┆ f64        ┆ f64        │
╞════════╪═════════════╪═══════════╪═════════╪═════════════╪═════════════╪════════════╪════════════╡
│ look=1 ┆ 1           ┆ 1449      ┆ 1       ┆ 0.461797    ┆ 0.517115    ┆ 0.394589   ┆ 0.23979    │
└────────┴─────────────┴───────────┴─────────┴─────────────┴─────────────┴────────────┴────────────

In [353]:
# SL: per-step training loss / MSE (dense per-gradient-step curves; requires step-level artifacts)
if sl is not None:
    try:
        display(sl.plot_vs_step("loss", title="SL: per-step loss", save_path=WRITEUP_ASSETS / "sl_per_step_loss.html"))
        display(sl.plot_vs_step("mse", title="SL: per-step MSE", save_path=WRITEUP_ASSETS / "sl_per_step_mse.html"))
    except ValueError as e:
        print(f"SL per-step plots unavailable (old-format artifacts?): {e}")

## SL+NTP-CE (token-level cross-entropy supervised)

Drop-in CE baseline for SL/MSE. Full-param fine-tunes the pretrained
`AutoModelForCausalLM` with cross-entropy on the lookahead token id (K=1
only). Validation reuses the projection from `pretrained_baseline.py`:
`softmax(logits) @ label_projector` → MSE, so `sl_ce` lands in the same
target space as SL/GRPO/RLOO/MaxRL and is directly comparable.

In [354]:
sl_ce = CorpusRegressionAnalysisConfig.from_sl_ce_sweep(
    artifacts_root=ARTIFACTS,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,
    lr_per_sample=LR_PER_SAMPLE,
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
)
if sl_ce is None:
    print("SL+NTP-CE: no artifacts")
else:
    sl_ce.describe("SL+NTP-CE")
    print(sl_ce.summarize(metric="corr"))
    print(sl_ce.summarize(metric="mse"))
    display(
        sl_ce.plot_vs_eval(
            "corr",
            title="SL+NTP-CE: per-eval (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_ce_per_eval_corr.html",
        )
    )
    display(
        sl_ce.plot_vs_eval(
            "mse",
            title="SL+NTP-CE: per-eval (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_ce_per_eval_mse.html",
        )
    )

SL+NTP-CE                            2 runs     1 groups   up to 2 seeds/group
shape: (1, 8)
┌────────┬─────────────┬───────────┬─────────┬─────────────┬─────────────┬────────────┬────────────┐
│ study  ┆ num_lookfor ┆ best_step ┆ n_seeds ┆ train_corr_ ┆ val_corr_me ┆ train_pred ┆ val_pred_s │
│ ---    ┆ ward        ┆ ---       ┆ ---     ┆ mean        ┆ an          ┆ _std_ratio ┆ td_ratio_m │
│ str    ┆ ---         ┆ i64       ┆ i64     ┆ ---         ┆ ---         ┆ _mean      ┆ ean        │
│        ┆ i64         ┆           ┆         ┆ f64         ┆ f64         ┆ ---        ┆ ---        │
│        ┆             ┆           ┆         ┆             ┆             ┆ f64        ┆ f64        │
╞════════╪═════════════╪═══════════╪═════════╪═════════════╪═════════════╪════════════╪════════════╡
│ look=1 ┆ 1           ┆ 1299      ┆ 2       ┆ 0.524875    ┆ 0.530751    ┆ 0.340582   ┆ 0.339957   │
└────────┴─────────────┴───────────┴─────────┴─────────────┴─────────────┴────────────┴────────────

In [355]:
# SL+NTP-CE: per-step training loss / MSE (CE loss; projected MSE diagnostic)
if sl_ce is not None:
    try:
        display(sl_ce.plot_vs_step("loss", title="SL+NTP-CE: per-step loss (CE)", save_path=WRITEUP_ASSETS / "sl_ce_per_step_loss.html"))
        display(sl_ce.plot_vs_step("mse", title="SL+NTP-CE: per-step MSE (projected)", save_path=WRITEUP_ASSETS / "sl_ce_per_step_mse.html"))
    except ValueError as e:
        print(f"SL+NTP-CE per-step plots unavailable (old-format artifacts?): {e}")

## GRPO

In [356]:
grpo = CorpusRegressionAnalysisConfig.from_grpo_sweep(
    artifacts_root=ARTIFACTS,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES, gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE, normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,
    lr_per_sample=LR_PER_SAMPLE,
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
)
if grpo is None:
    print("GRPO: no artifacts")
else:
    grpo.describe("GRPO")
    print(grpo.summarize(metric="corr"))
    print(grpo.summarize(metric="mse"))
    display(
        grpo.plot_vs_eval(
            "corr",
            title="GRPO: per-eval (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_eval_corr.html",
        )
    )
    display(
        grpo.plot_vs_eval(
            "mse",
            title="GRPO: per-eval (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_eval_mse.html",
        )
    )

GRPO                                 4 runs     2 groups   up to 2 seeds/group
shape: (2, 8)
┌────────┬─────────────┬───────────┬─────────┬─────────────┬─────────────┬────────────┬────────────┐
│ study  ┆ num_lookfor ┆ best_step ┆ n_seeds ┆ train_corr_ ┆ val_corr_me ┆ train_pred ┆ val_pred_s │
│ ---    ┆ ward        ┆ ---       ┆ ---     ┆ mean        ┆ an          ┆ _std_ratio ┆ td_ratio_m │
│ str    ┆ ---         ┆ i64       ┆ i64     ┆ ---         ┆ ---         ┆ _mean      ┆ ean        │
│        ┆ i64         ┆           ┆         ┆ f64         ┆ f64         ┆ ---        ┆ ---        │
│        ┆             ┆           ┆         ┆             ┆             ┆ f64        ┆ f64        │
╞════════╪═════════════╪═══════════╪═════════╪═════════════╪═════════════╪════════════╪════════════╡
│ look=1 ┆ 1           ┆ 1499      ┆ 2       ┆ 0.4281      ┆ 0.501401    ┆ 0.362553   ┆ 0.187867   │
│ r=16   ┆             ┆           ┆         ┆             ┆             ┆            ┆            

In [357]:
# GRPO: per-step training loss / MSE
if grpo is not None:
    try:
        display(grpo.plot_vs_step("loss", title="GRPO: per-step loss", save_path=WRITEUP_ASSETS / "grpo_per_step_loss.html"))
        display(grpo.plot_vs_step("mse", title="GRPO: per-step MSE", save_path=WRITEUP_ASSETS / "grpo_per_step_mse.html"))
    except ValueError as e:
        print(f"GRPO per-step plots unavailable (old-format artifacts?): {e}")

## MaxRL (subtract-baseline + factorized)

In [358]:
maxrl_sf = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
    artifacts_root=ARTIFACTS,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,
    lr_per_sample=LR_PER_SAMPLE,
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
)
if maxrl_sf is None:
    print("MaxRL (sub-baseline, factorized): no artifacts")
else:
    maxrl_sf.describe("MaxRL (sub-baseline, factorized)")
    print(maxrl_sf.summarize(metric="corr"))
    print(maxrl_sf.summarize(metric="mse"))
    display(
        maxrl_sf.plot_vs_eval(
            "corr",
            title="MaxRL (sub-baseline, factorized): per-eval (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_eval_corr.html",
        )
    )
    display(
        maxrl_sf.plot_vs_eval(
            "mse",
            title="MaxRL (sub-baseline, factorized): per-eval (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_eval_mse.html",
        )
    )

MaxRL (sub-baseline, factorized)     4 runs     2 groups   up to 2 seeds/group
shape: (2, 8)
┌────────┬─────────────┬───────────┬─────────┬─────────────┬─────────────┬────────────┬────────────┐
│ study  ┆ num_lookfor ┆ best_step ┆ n_seeds ┆ train_corr_ ┆ val_corr_me ┆ train_pred ┆ val_pred_s │
│ ---    ┆ ward        ┆ ---       ┆ ---     ┆ mean        ┆ an          ┆ _std_ratio ┆ td_ratio_m │
│ str    ┆ ---         ┆ i64       ┆ i64     ┆ ---         ┆ ---         ┆ _mean      ┆ ean        │
│        ┆ i64         ┆           ┆         ┆ f64         ┆ f64         ┆ ---        ┆ ---        │
│        ┆             ┆           ┆         ┆             ┆             ┆ f64        ┆ f64        │
╞════════╪═════════════╪═══════════╪═════════╪═════════════╪═════════════╪════════════╪════════════╡
│ look=1 ┆ 1           ┆ 1499      ┆ 2       ┆ 0.450704    ┆ 0.506062    ┆ 0.369592   ┆ 0.199048   │
│ r=16   ┆             ┆           ┆         ┆             ┆             ┆            ┆            

In [359]:
# MaxRL: per-step training loss / MSE
if maxrl_sf is not None:
    try:
        display(maxrl_sf.plot_vs_step("loss", title="MaxRL (sub-baseline, factorized): per-step loss", save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_step_loss.html"))
        display(maxrl_sf.plot_vs_step("mse", title="MaxRL (sub-baseline, factorized): per-step MSE", save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_step_mse.html"))
    except ValueError as e:
        print(f"MaxRL per-step plots unavailable (old-format artifacts?): {e}")

### Other MaxRL ablations

Uncomment to inspect the other (`subtract_baseline`, `use_factorized_likelihoods`) combinations if we run these sweeps

In [360]:
# for sub, fact in [(True, False), (False, True), (False, False)]:
#     cfg = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
#         artifacts_root=ARTIFACTS,
#         subtract_baseline=sub,
#         use_factorized_likelihoods=fact,
#         num_samples=NUM_SAMPLES,
#         gaussian_stdev=GAUSSIAN_STDEV,
#     )
#     label = f"MaxRL (sub={sub}, fact={fact})"
#     if cfg is None:
#         print(f"{label}: no artifacts")
#         continue
#     cfg.describe(label)
#     display(cfg.plot_vs_eval("corr", title=f"{label}: per-eval (corr)", show_seed_bar=True))
#     display(cfg.plot_vs_lookforward(title=f"{label}: best-step vs num_lookforward_tokens", x_scale="uniform", show_seed_bar=True))

## RLOO (factorized)

In [361]:
rloo_f = CorpusRegressionAnalysisConfig.from_rloo_sweep(
    artifacts_root=ARTIFACTS,
    factorized=True,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,
    lr_per_sample=LR_PER_SAMPLE,
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
)
if rloo_f is None:
    print("RLOO (factorized): no artifacts")
else:
    rloo_f.describe("RLOO (factorized)")
    print(rloo_f.summarize(metric="corr"))
    print(rloo_f.summarize(metric="mse"))
    display(
        rloo_f.plot_vs_eval(
            "corr",
            title="RLOO (factorized): per-eval (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_eval_corr.html",
        )
    )
    display(
        rloo_f.plot_vs_eval(
            "mse",
            title="RLOO (factorized): per-eval (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_eval_mse.html",
        )
    )

RLOO (factorized)                    4 runs     2 groups   up to 2 seeds/group
shape: (2, 8)
┌────────┬─────────────┬───────────┬─────────┬─────────────┬─────────────┬────────────┬────────────┐
│ study  ┆ num_lookfor ┆ best_step ┆ n_seeds ┆ train_corr_ ┆ val_corr_me ┆ train_pred ┆ val_pred_s │
│ ---    ┆ ward        ┆ ---       ┆ ---     ┆ mean        ┆ an          ┆ _std_ratio ┆ td_ratio_m │
│ str    ┆ ---         ┆ i64       ┆ i64     ┆ ---         ┆ ---         ┆ _mean      ┆ ean        │
│        ┆ i64         ┆           ┆         ┆ f64         ┆ f64         ┆ ---        ┆ ---        │
│        ┆             ┆           ┆         ┆             ┆             ┆ f64        ┆ f64        │
╞════════╪═════════════╪═══════════╪═════════╪═════════════╪═════════════╪════════════╪════════════╡
│ look=1 ┆ 1           ┆ 1349      ┆ 2       ┆ 0.415914    ┆ 0.497643    ┆ 0.456369   ┆ 0.213247   │
│ r=16   ┆             ┆           ┆         ┆             ┆             ┆            ┆            

In [362]:
# RLOO: per-step training loss / MSE
if rloo_f is not None:
    try:
        display(rloo_f.plot_vs_step("loss", title="RLOO (factorized): per-step loss", save_path=WRITEUP_ASSETS / "rloo_factorized_per_step_loss.html"))
        display(rloo_f.plot_vs_step("mse", title="RLOO (factorized): per-step MSE", save_path=WRITEUP_ASSETS / "rloo_factorized_per_step_mse.html"))
    except ValueError as e:
        print(f"RLOO per-step plots unavailable (old-format artifacts?): {e}")

## Pretrained baseline (intrinsic-variance proxy)

Inference-only: deterministic given the dataset (no seeds, single step).
We assemble one config across `candidate_lookforward_tokens` so it slots
into the same plotting helpers as the trained methods.

In [363]:
# Pretrained baseline now accepts a list of lookforwards directly (no manual merge).
# Sweeping `candidate_lookforward_tokens` here so plot_vs_lookforward keeps a
# multi-point x-axis even though the per-method plots above use a single look.
ntp = CorpusRegressionAnalysisConfig.from_pretrained_baseline(
    artifacts_root=ARTIFACTS,
    num_lookforward_tokens=candidate_lookforward_tokens,
    num_samples=NUM_SAMPLES,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
)

if ntp is None:
    print("Pretrained baseline: no artifacts")
else:
    ntp.describe("Pretrained baseline")
    print(ntp.summarize(metric="corr"))
    print(ntp.summarize(metric="mse"))
    display(
        ntp.plot_vs_lookforward(
            title="Pretrained baseline: best-step vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=False,
            save_path=WRITEUP_ASSETS / "pretrained_baseline_vs_lookforward.html",
        )
    )

Pretrained baseline: no artifacts


## Constant-predictor MSE baseline (target variance)

For a constant predictor $\hat y \equiv c$, eval MSE decomposes as
$\text{MSE} = \text{Var}_{\text{eval}}(y) + (c - \bar y_{\text{eval}})^2$.
So `val_target_var` (= `val_target_std`$^2$) is the MSE floor of the best
constant predictor on the eval set, and is within $O(1/n)$ of the MSE of
the empirical-train-mean predictor for iid splits. Use these numbers as a
reference: a method whose val_mse plateaus near `val_target_var` is
essentially predicting a constant.

`val_target_var` is a property of the fixed eval set, so its std across
rows should be near zero. `train_target_var` is a *training-window*
aggregate for trained methods (will vary mildly across rows); for the
`pretrained` baseline it is a full train-pass aggregate.

In [364]:
# Per-method summary of train/val target variance — i.e. the MSE floor of
# the best constant predictor (≈ MSE of the empirical-train-mean baseline).
def _target_var_summary(name, cfg):
    if cfg is None:
        return None
    df = cfg.get_metric_dataframe()
    row = {"method": name, "n_rows": df.height}
    for split in ("train", "val"):
        col = f"{split}_target_var"
        vals = df[col].drop_nulls() if col in df.columns else pl.Series([], dtype=pl.Float64)
        if vals.len() == 0:
            row[f"{split}_target_var_mean"] = None
            row[f"{split}_target_var_std"] = None
            row[f"{split}_target_std_mean"] = None
            continue
        mean_var = float(vals.mean())
        std_var = vals.std()
        row[f"{split}_target_var_mean"] = mean_var
        row[f"{split}_target_var_std"] = float(std_var) if std_var is not None else 0.0
        row[f"{split}_target_std_mean"] = mean_var ** 0.5
    return row

_baseline_rows = [
    r for r in (
        _target_var_summary("sl", sl),
        _target_var_summary("sl_ce", sl_ce),
        _target_var_summary("grpo", grpo),
        _target_var_summary("maxrl_sf", maxrl_sf),
        _target_var_summary("rloo_f", rloo_f),
        _target_var_summary("pretrained", ntp),
    )
    if r is not None
]
if _baseline_rows:
    print(
        "Constant-predictor MSE floor (≈ {split}_target_var, dim-averaged, "
        "averaged across rows). "
        "target_std_mean is sqrt of target_var_mean for reference."
    )
    print(pl.DataFrame(_baseline_rows))
else:
    print("No configs loaded; skipping target-variance baseline summary.")

Constant-predictor MSE floor (≈ {split}_target_var, dim-averaged, averaged across rows). target_std_mean is sqrt of target_var_mean for reference.
shape: (5, 8)
┌──────────┬────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ method   ┆ n_rows ┆ train_targe ┆ train_targ ┆ train_targ ┆ val_target ┆ val_target ┆ val_target │
│ ---      ┆ ---    ┆ t_var_mean  ┆ et_var_std ┆ et_std_mea ┆ _var_mean  ┆ _var_std   ┆ _std_mean  │
│ str      ┆ i64    ┆ ---         ┆ ---        ┆ n          ┆ ---        ┆ ---        ┆ ---        │
│          ┆        ┆ f64         ┆ f64        ┆ ---        ┆ f64        ┆ f64        ┆ f64        │
│          ┆        ┆             ┆            ┆ f64        ┆            ┆            ┆            │
╞══════════╪════════╪═════════════╪════════════╪════════════╪════════════╪════════════╪════════════╡
│ sl       ┆ 30     ┆ 0.02782     ┆ 0.000919   ┆ 0.166792   ┆ 0.027811   ┆ 0.0        ┆ 0.166767   │
│ sl_ce    ┆ 60     ┆ 0.027845 

## Cross-method comparison

In [365]:
# if any(c is not None for c in (sl, sl_ce, grpo, maxrl_sf, rloo_f, ntp)):
#     display(
#         plot_methods_vs_lookforward(
#             sl_mse=sl,
#             sl_ce=sl_ce,
#             grpo=grpo,
#             maxrl=maxrl_sf,
#             rloo=rloo_f,
#             pretrained_baseline=ntp,
#             title="Methods: best-step corr vs num_lookforward_tokens",
#             x_scale="uniform",
#             save_path=WRITEUP_ASSETS / "methods_vs_lookforward.html",
#         )
#     )
#     display(
#         plot_methods_vs_lookforward(
#             sl_mse=sl,
#             sl_ce=sl_ce,
#             grpo=grpo,
#             maxrl=maxrl_sf,
#             rloo=rloo_f,
#             pretrained_baseline=ntp,
#             metric="mse",
#             title="Methods: best-step MSE vs num_lookforward_tokens",
#             x_scale="uniform",
#             save_path=WRITEUP_ASSETS / "methods_vs_lookforward_mse.html",
#         )
#     )
# else:
#     print("No artifacts for any method.")

### Per-eval training curves (look=1, all methods)

In [366]:
if any(c is not None for c in (sl, sl_ce, grpo, maxrl_sf, rloo_f, ntp)):
    display(
        plot_methods_vs_eval(
            sl_mse=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            pretrained_baseline=ntp,
            num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
            metric="corr",
            show_seed_bar=True,
            title=f"Methods: per-eval corr (look={NUM_LOOKFORWARD_TOKENS})",
            save_path=WRITEUP_ASSETS / "methods_vs_eval_corr.html",
        )
    )
    display(
        plot_methods_vs_eval(
            sl_mse=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            pretrained_baseline=ntp,
            num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
            metric="mse",
            show_seed_bar=True,
            title=f"Methods: per-eval MSE (look={NUM_LOOKFORWARD_TOKENS})",
            save_path=WRITEUP_ASSETS / "methods_vs_eval_mse.html",
        )
    )
else:
    print("No artifacts for any method.")

### Collapse diagnostic: prediction-std ratio (look=1, all methods)

`pred_std / target_std` per eval step. A ratio approaching 0 is the
regression-collapse signature: the model is learning a near-constant
output (variance shrinking toward zero) regardless of input. A ratio
≈ 1 means the prediction's spread matches the target's. This requires
the new `*_pred_sum` / `*_target_sum` columns in `val_metrics.parquet`;
runs from before this schema change won't appear here.

In [367]:
if any(c is not None for c in (sl, sl_ce, grpo, maxrl_sf, rloo_f, ntp)):
    display(
        plot_methods_pred_std_vs_eval(
            sl_mse=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            pretrained_baseline=ntp,
            num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
            quantity="pred_std_ratio",
            show_seed_bar=True,
            title=f"Methods: pred_std / target_std (collapse diagnostic, look={NUM_LOOKFORWARD_TOKENS})",
            save_path=WRITEUP_ASSETS / "methods_pred_std_ratio.html",
        )
    )
    display(
        plot_methods_pred_std_vs_eval(
            sl_mse=sl,
            sl_ce=sl_ce,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            pretrained_baseline=ntp,
            num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
            quantity="pred_std",
            show_seed_bar=True,
            title=f"Methods: pred_std (raw, look={NUM_LOOKFORWARD_TOKENS})",
            save_path=WRITEUP_ASSETS / "methods_pred_std.html",
        )
    )
else:
    print("No artifacts for any method.")

## Group-axis examples (new)

Per-method `plot_vs_eval` / `plot_vs_step` now accept a configurable group
axis. Pass any one of `num_samples`, `lr_per_sample`, `train_from_scratch`,
or `gaussian_stdev` (RL only) as a *list* — the curves will be grouped
(legend / colour) by that axis instead of by lookforward.

Example: compare `num_samples ∈ {100k, 500k}` for SL at the current look.

In [368]:
maxrl_by_n = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
    artifacts_root=ARTIFACTS,
    num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
    num_samples=NUM_SAMPLES,   # ← list ⇒ this becomes the group axis
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
    batch_size=BATCH_SIZE,    # pass a list (e.g. [64, 128]) to group by batch_size instead
    lr_per_sample=[2e-4,1e-4,5e-5,2e-5,1e-5,5e-6],
    lr_schedule=LR_SCHEDULE, warmup_ratio=WARMUP_RATIO, lr_min_ratio=LR_MIN_RATIO,
    train_from_scratch=TRAIN_FROM_SCRATCH,
    train_steps=TRAIN_STEPS,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
)
if maxrl_by_n is None:
    print("MaxRL grouped by num_samples: no artifacts")
else:
    maxrl_by_n.describe(f"MaxRL by num_samples (group_by={maxrl_by_n.group_by!r})")
    print(maxrl_by_n.summarize(metric="mse"))
    display(
        maxrl_by_n.plot_vs_eval(
            "mse",
            num_lookforward_tokens=NUM_LOOKFORWARD_TOKENS,
            title=f"MaxRL: per-eval corr by num_samples (look={NUM_LOOKFORWARD_TOKENS})",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrk_per_eval_corr_group.html",
        )
    )

MaxRL by num_samples (group_by='lr_per_sample')    12 runs     6 groups   up to 2 seeds/group
shape: (6, 8)
┌────────────┬────────────┬───────────┬─────────┬────────────┬────────────┬────────────┬───────────┐
│ study      ┆ num_lookfo ┆ best_step ┆ n_seeds ┆ train_mse_ ┆ val_mse_me ┆ train_pred ┆ val_pred_ │
│ ---        ┆ rward      ┆ ---       ┆ ---     ┆ mean       ┆ an         ┆ _std_ratio ┆ std_ratio │
│ str        ┆ ---        ┆ i64       ┆ i64     ┆ ---        ┆ ---        ┆ _mean      ┆ _mean     │
│            ┆ i64        ┆           ┆         ┆ f64        ┆ f64        ┆ ---        ┆ ---       │
│            ┆            ┆           ┆         ┆            ┆            ┆ f64        ┆ f64       │
╞════════════╪════════════╪═══════════╪═════════╪════════════╪════════════╪════════════╪═══════════╡
│ look=1 lr= ┆ 1          ┆ 1249      ┆ 2       ┆ 0.028488   ┆ 0.027179   ┆ 0.222055   ┆ 0.132634  │
│ 1.00e-04   ┆            ┆           ┆         ┆            ┆            ┆         